In [1]:
# folder with text files
folder_with_text = './text'

# max text length
max_length=1024

# one token embedding
emb_dim=32

# latent space z size
internal_dim=32

# training batch size
batch_size=128

# where to save model checkpoints
checkpoint_path = 'runs/test-autoregressive'

# number of epochs to train
num_epochs=500

# Create tokenizer

In [2]:
import numpy as np
import torch
import torch.nn as nn
from torch import Tensor
from typing import List, Dict
import os
from kemsekov_torch.text_tools import SimpleTokenizer


txt_files = [[os.path.join(fdir,f) for f in files if f.endswith(".txt")] for fdir,_,files in os.walk(folder_with_text)]
txt_files = [b for a in txt_files for b in a]
txt_lines = [open(v).read() for v in txt_files]

tokenizer = SimpleTokenizer(txt_lines,lowercase=True,unknown_symbols_placeholder=' ')
torch.jit.script(tokenizer).save("tokenizer.pt")

test_str="This is my TEST string! Раз!"
inds=tokenizer.encode(test_str)
print(test_str)
print(inds)
print(tokenizer.decode(inds))

Text length analysis
text lines	 79295
line chars mean	 78.267
line chars std	 180.905
0.05 quantile	 0.0
0.95 quantile	 341.0
0.995 quantile	 711.0
This is my TEST string! Раз!
tensor([54, 42, 43, 53,  1, 43, 53,  1, 47, 59,  1, 54, 39, 53, 54,  1, 53, 54,
        52, 43, 48, 41,  2,  1,  1,  1,  1,  2])
this is my test string!    !


/home/vlad/Programs/venv/lib/python3.14/site-packages/torch/jit/_script.py:1485: FutureWarning: `torch.jit.script` is not supported in Python 3.14+ and may break. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


# Define Dataset

In [3]:
import math
from kemsekov_torch.train import split_dataset
import torch
from kemsekov_torch.text_tools import TokenDataset

txt_split = [b for v in txt_lines if len(v)>30 for b in v.split('\n')]
dataset = TokenDataset(
    tokenizer,
    txt_split[:15000],
    pad_token=tokenizer.unknown_symbols_placeholder,
    batch_size=batch_size,
    max_length=max_length
)

# split dataset into train and test
train_dataset,test_dataset,train_loader, test_loader = split_dataset(
    dataset,
    test_size=0.05,
    batch_size=batch_size,
    random_state=None,
    bin_by_size=True,
    num_workers=1,
)

Bin by tensor size: 100%|██████████| 6859/6859 [00:05<00:00, 1285.21it/s]


Unique 6859
Total 7424
Bins 8


Bin by tensor size: 100%|██████████| 362/362 [00:00<00:00, 774.98it/s]


Unique 337
Total 512
Bins 3
Train items 7424
Test items 512


In [ ]:
for t in train_loader:
    print(t.shape)
    pass

torch.Size([128, 384])
torch.Size([128, 384])
torch.Size([128, 384])
torch.Size([128, 384])
torch.Size([128, 384])
torch.Size([128, 384])
torch.Size([128, 256])
torch.Size([128, 256])
torch.Size([128, 256])
torch.Size([128, 256])
torch.Size([128, 256])
torch.Size([128, 256])
torch.Size([128, 256])
torch.Size([128, 256])
torch.Size([128, 256])
torch.Size([128, 256])
torch.Size([128, 256])
torch.Size([128, 256])
torch.Size([128, 256])
torch.Size([128, 256])
torch.Size([128, 256])
torch.Size([128, 128])
torch.Size([128, 128])
torch.Size([128, 128])
torch.Size([128, 128])
torch.Size([128, 128])
torch.Size([128, 128])
torch.Size([128, 128])
torch.Size([128, 128])
torch.Size([128, 128])
torch.Size([128, 128])
torch.Size([128, 128])
torch.Size([128, 128])
torch.Size([128, 128])
torch.Size([128, 128])
torch.Size([128, 128])
torch.Size([128, 128])
torch.Size([128, 128])
torch.Size([128, 128])
torch.Size([128, 128])
torch.Size([128, 128])
torch.Size([128, 128])
torch.Size([128, 128])
torch.Size(

In [9]:
import random
ind = random.randint(0,len(train_dataset)-1)
inds = dataset[ind]

print("Text length",len(inds))
skip_text = tokenizer.decode(inds).strip()
print(skip_text)

Text length 128
harry started to explain in a whisper, but at that moment the headmaster stood up to speak, and he broke off.


# Define Model

In [10]:
from kemsekov_torch.residual import ResidualBlock
from kemsekov_torch.common_modules import *
import torch
import torch.nn as nn

# module to convert text tokens to vector
class Embedding(nn.Module):
    """
    Module for token to embedding vector learning
    """
    def __init__(self, vocab_size, embedding_size):
        super().__init__()
        self.vocab_size = vocab_size
        self.embedding_size = embedding_size

        # Initialize weights and bias
        self.weight = nn.Parameter(torch.Tensor(vocab_size, embedding_size))
        self.bias = nn.Parameter(torch.Tensor(embedding_size))

        self.reset_parameters()

    #normal init
    def reset_parameters(self):
        # Initialize weights with a normal distribution
        std = 1.0 / (self.vocab_size**0.5)
        nn.init.normal_(self.weight, mean=0.0, std=std)
        # Initialize bias to zeros
        nn.init.zeros_(self.bias)
        
    def forward(self, input):
        # Input is expected to be a tensor of indices
        output = torch.nn.functional.embedding(input, self.weight) + self.bias
        return output.transpose(-1,-2)

# Training

In [ ]:
# here is what we do here
# 1. convert text to token ids
#   "this is text" -> [0,2,5,1,2,4,8,4]
# 2. convert ids to vectors via embedding
#   [0,2,5,1,2,4,8,4] -> Tensor((8,32))
# 3. Collapse tensor to latent space via vae
#   Tensor((8,32)) -> mean,std
#   z=randn()*mean+std
# 4. Apply l2 regularization and curvature loss to latent space z
#   loss = l2(z)+(d/dt(z))^2
# 5. train model to reconstruct original tensor
#   z -> Tensor((8,32))
# 6. Convert tensor back to original tokens
#   Tensor((8,32)) -> [0,2,5,1,2,4,8,4]

from kemsekov_torch.train import train
from torchmetrics.classification import MulticlassF1Score

CE = torch.nn.CrossEntropyLoss()

f1__ = MulticlassF1Score(tokenizer.vocab_size)
def f1(x,y):
    return f1__(x.detach().cpu(),y.detach().cpu())

def first_order_loss(z):
    diff = z[:, 1:] - z[:, :-1]
    scale = z.abs().mean(-2,keepdim=True).detach()+1e-6
    return (diff.pow(2)/scale).mean()


def curvature_loss(z):
    """
    Loss function designed to make a latent space smooth
    """
    return first_order_loss(z)

def compute_loss_and_metric(model,batch):
    text = batch[0]

    recon,z_mean,z_std = model(text)
    recon=recon.transpose(-2,-1)
    
    smoothness_loss = curvature_loss(z_mean)
    l2_loss = (z_mean).pow(2).mean()
    # z embedding [batch,tokens_count/4,z_emb_dim]
    
    reconstruction_loss = CE(recon,text)
    
    # use two losses, to enforce model not only reconstruct back original
    # input sequence, but also pay same amount of attention to skipped tokens
    return reconstruction_loss+0.1*(smoothness_loss+l2_loss),{
        'reconstruction_f1': f1(recon,text),
        'reconstruction_loss':reconstruction_loss,
        'smoothness_loss':smoothness_loss,
        'l2_loss':l2_loss
    }


optim = torch.optim.AdamW(model.parameters(),0.001,betas=(0.9, 0.95))
sch = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optim,len(train_loader))

_ = train(
    model,
    train_loader,
    test_loader,
    compute_loss_and_metric,
    checkpoint_path,
    # f'{checkpoint_path}/last',
    gradient_clipping_max_norm=1,
    accelerate_args={
        'mixed_precision':'bf16',
        'dynamo_backend':'inductor'
    },
    save_on_metric_improve=['reconstruction_f1'],
    optimizer=optim,
    num_epochs=num_epochs,
    scheduler=sch,
    checkpoints_count=1,
)

Total model parameters 0.07 M
trying to capture model architecture...
Saved model architecture at runs/text-autoencoder/model.pt. You can torch.load it and update it's weights with checkpoint

Epoch 1/500


train 0: 100%|██████████| 1599/1599 [00:29<00:00, 53.37it/s, l2_loss=0.0171, loss=0.2269, reconstruction_f1=0.5723, reconstruction_loss=0.2197, smoothness_loss=0.0554] 


+---------------------+---------+---------+
|                     |  Train  |  Test   |
+---------------------+---------+---------+
|        loss         | 0.75162 | 0.14978 |
|  reconstruction_f1  | 0.4848  | 0.7476  |
| reconstruction_loss | 0.7335  | 0.1440  |
|   smoothness_loss   | 0.0597  | 0.0420  |
|       l2_loss       | 0.1213  | 0.0163  |
+---------------------+---------+---------+
saved epoch-1

Epoch 2/500


train 0: 100%|██████████| 1599/1599 [00:15<00:00, 100.57it/s, l2_loss=0.0015, loss=0.0261, reconstruction_f1=0.7690, reconstruction_loss=0.0223, smoothness_loss=0.0372]


+---------------------+---------+--------+
|                     |  Train  |  Test  |
+---------------------+---------+--------+
|        loss         | 0.07057 | 0.0165 |
|  reconstruction_f1  | 0.9003  | 0.9453 |
| reconstruction_loss | 0.0633  | 0.0135 |
|   smoothness_loss   | 0.0673  | 0.0287 |
|       l2_loss       | 0.0059  | 0.0015 |
+---------------------+---------+--------+
saved epoch-2

Epoch 3/500


...

+---------------------+---------+--------+
|                     |  Train  |  Test  |
+---------------------+---------+--------+
|        loss         | 0.07357 | 0.0074 |
|  reconstruction_f1  | 0.9463  | 0.9691 |
| reconstruction_loss | 0.0680  | 0.0052 |
|   smoothness_loss   | 0.0536  | 0.0215 |
|       l2_loss       | 0.0022  | 0.0007 |
+---------------------+---------+--------+
saved epoch-3

Epoch 4/500


train 0: 100%|██████████| 1599/1599 [00:16<00:00, 96.04it/s, l2_loss=0.0005, loss=0.0077, reconstruction_f1=0.7773, reconstruction_loss=0.0052, smoothness_loss=0.0246] 


+---------------------+--------+---------+
|                     | Train  |  Test   |
+---------------------+--------+---------+
|        loss         | 0.0717 | 0.00549 |
|  reconstruction_f1  | 0.9560 | 0.9775  |
| reconstruction_loss | 0.0668 | 0.0035  |
|   smoothness_loss   | 0.0473 | 0.0192  |
|       l2_loss       | 0.0016 | 0.0005  |
+---------------------+--------+---------+
saved epoch-4

Epoch 5/500


train 0:  16%|█▋        | 261/1599 [00:02<00:12, 110.24it/s, l2_loss=0.0025, loss=0.0174, reconstruction_f1=0.9747, reconstruction_loss=0.0093, smoothness_loss=0.0783]


Interrupt training
